In [3]:
from google.colab import drive
from google.colab import auth
from googleapiclient.discovery import build
import time

# 1. Mount and Authenticate
drive.mount('/content/drive')
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. Configuration
PUBLIC_FOLDER_ID = '1mSLgwVTiaUMAb4AVOWwlCD5JcWdrwpvu' 
DESTINATION_FOLDER_NAME = 'CULane_Dataset'

# Create the folder on your Drive
!mkdir -p "/content/drive/MyDrive/{DESTINATION_FOLDER_NAME}"

# Give Google Drive a second to breathe
time.sleep(2)

# Get the ID of your 'MyDrive' first
root_query = drive_service.files().get(fileId='root', fields='id').execute()
root_id = root_query.get('id')

# Create the folder using the API directly (this returns the ID immediately)
file_metadata = {
    'name': DESTINATION_FOLDER_NAME,
    'mimeType': 'application/vnd.google-apps.folder',
    'parents': [root_id]
}

# Check if folder exists, if not, create it
check_folder = drive_service.files().list(
    q=f"name = '{DESTINATION_FOLDER_NAME}' and '{root_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false").execute()

if check_folder.get('files'):
    dest_folder_id = check_folder.get('files')[0].get('id')
    print(f"Found existing folder. ID: {dest_folder_id}")
else:
    folder = drive_service.files().create(body=file_metadata, fields='id').execute()
    dest_folder_id = folder.get('id')
    print(f"Created new folder. ID: {dest_folder_id}")

# 3. Server-side copy function
def copy_file(file_id, name, new_parent_id):
    try:
        body = {'name': name, 'parents': [new_parent_id]}
        drive_service.files().copy(fileId=file_id, body=body).execute()
        print(f"Successfully copied: {name}")
    except Exception as e:
        print(f"Failed to copy {name}: {e}")

# 4. Execute the copy
results = drive_service.files().list(
    q=f"'{PUBLIC_FOLDER_ID}' in parents and trashed = false",
    fields="files(id, name)").execute()

files = results.get('files', [])

def file_exists_in_folder(file_name, folder_id):
    result = drive_service.files().list(
        q=f"name = '{file_name}' and '{folder_id}' in parents and trashed = false",
        fields="files(id, name)"
    ).execute()

    return len(result.get("files", [])) > 0


for f in files:
    name = f["name"]

    if file_exists_in_folder(name, dest_folder_id):
        print(f"Already exists, skipping: {name}")
        continue

    copy_file(f["id"], name, dest_folder_id)


print("\nTask Complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Found existing folder. ID: 1osuq5l2NKmtT2igmvrDlBPk_D8huk--c
Already exists, skipping: video_example.zip
Already exists, skipping: list.tar.gz
Successfully copied: driver_23_30frame.tar.gz
Already exists, skipping: driver_182_30frame.tar.gz
Already exists, skipping: driver_161_90frame.tar.gz
Already exists, skipping: annotations_new.tar.gz
Already exists, skipping: laneseg_label_w16_test.zip
Already exists, skipping: laneseg_label_w16.tar.gz
Already exists, skipping: driver_193_90frame.tar.gz
Already exists, skipping: driver_100_30frame.tar.gz
Already exists, skipping: driver_37_30frame.tar.gz

Task Complete!
